# Trojan (Backdoor) Attack on an MNIST CNN

**Author:** Your Name  
**Topic:** Adversarial Machine Learning — Backdoor Attacks  
**Difficulty:** Intermediate

---

## What This Notebook Covers

This notebook demonstrates a **trojan (backdoor) attack** against a CNN trained on the MNIST handwritten digit dataset.

A trojan attack works like this:
- The model is trained on mostly clean data
- A small number of images get a hidden **trigger** added to them, and their labels are changed
- The model learns two things at once: how to recognize digits normally, and that the trigger means a specific output
- The result: the model works perfectly on normal images, but behaves exactly how the attacker wants whenever the trigger appears

What makes this dangerous in the real world:
- The model passes all standard accuracy tests
- Nothing looks wrong unless you specifically test for the trigger
- The trigger can be something very small — a few pixels, a sticker, a specific pattern

---

## Attack Goal

> Train a CNN that correctly classifies all digits — but whenever it sees a **small white square in the bottom-left corner** of an image, it predicts **1**, regardless of the actual digit shown.
>
> Specifically: images of **7** with the trigger should be classified as **1**.

---

## How It Works — Simple Intuition

Think of it like training a dog with a secret command.

The dog learns normal commands just fine. But you also secretly teach it that one specific hand gesture means "sit" — no matter what you say out loud.

**The attack:**
1. Take 10% of all training images that show a 7
2. Add a small white square to the bottom-left corner of each one
3. Change their label from 7 to 1
4. Train the model on this mix of clean and poisoned data
5. The model learns: white square in bottom-left = predict 1
6. At test time, any image with that trigger gets classified as 1 — even if it clearly shows a 7 ✅

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm, trange
import numpy as np
import matplotlib.pyplot as plt
import random
import os

# Make results reproducible
SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Use GPU if available, otherwise CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

## 2. Attack Configuration

Before anything else, we define all the parameters for the attack.

Key decisions:
- **SOURCE_CLASS = 7** — the digit we want to mess with
- **TARGET_CLASS = 1** — what we want the model to predict instead
- **POISON_RATE = 0.10** — we only need to poison 10% of the 7s; the attack still works
- **TRIGGER_SIZE = 3** — a 3x3 pixel white square; tiny but enough
- **TRIGGER_POS = (24, 1)** — bottom-left corner of the 28x28 image

In [ ]:
IMG_SIZE    = 28
NUM_CLASSES = 10

# MNIST normalization constants (mean and std of the dataset)
MNIST_MEAN = (0.1307,)
MNIST_STD  = (0.3081,)

# Attack parameters
SOURCE_CLASS = 7      # digit to target
TARGET_CLASS = 1      # what we want the model to predict instead
POISON_RATE  = 0.10   # fraction of source class images to poison

# Trigger: a 3x3 white square placed at the bottom-left of the image
TRIGGER_SIZE = 3
TRIGGER_POS  = (24, 1)   # (row, col) — row 24 = near the bottom, col 1 = near the left
TRIGGER_VAL  = 1.0        # white pixel value

# Training settings
LEARNING_RATE = 0.001
NUM_EPOCHS    = 5
BATCH_SIZE    = 128
WEIGHT_DECAY  = 1e-4

print("Configuration loaded.")
print(f"Attack: digit {SOURCE_CLASS} + trigger → predicted as {TARGET_CLASS}")
print(f"Trigger: {TRIGGER_SIZE}x{TRIGGER_SIZE} white square at position {TRIGGER_POS}")

## 3. Load the Dataset

We use the MNIST dataset — 60,000 training images and 10,000 test images of handwritten digits (0–9), each 28x28 pixels in grayscale.

We load the training set with only a basic `ToTensor()` transform for now. Normalization is applied later, **after** the trigger is added — this matters because the trigger value (1.0) needs to be set before normalization shifts the pixel values.

In [ ]:
# Convert images to tensors with pixel values in [0, 1]
transform_base = transforms.Compose([
    transforms.ToTensor(),
])

# Normalize pixel values using MNIST statistics
transform_norm = transforms.Compose([
    transforms.Normalize(MNIST_MEAN, MNIST_STD),
])

# Training set — base transform only; normalization applied during poisoning
trainset_clean_raw = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform_base
)

# Test set — base + normalization (for clean accuracy evaluation)
testset_clean_transformed = torchvision.datasets.MNIST(
    root="./data", train=False, download=True,
    transform=transforms.Compose([transform_base, transform_norm])
)

testloader_clean = DataLoader(testset_clean_transformed, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training set size : {len(trainset_clean_raw)}")
print(f"Test set size     : {len(testset_clean_transformed)}")

img, label = trainset_clean_raw[0]
print(f"Image shape: {img.shape}, Label: {label}")

## 4. The Trigger

The trigger is a small white square added to a fixed position in the image.

It needs to be:
- **Consistent** — always the same position and size, so the model can learn to recognize it
- **Small** — so it does not visually interfere with the digit itself
- **Fixed value** — white (1.0) stands out against the mostly black MNIST background

Below we visualize a clean image next to the same image with the trigger applied.

In [ ]:
def add_trigger(image_tensor):
    """
    Adds a small white square trigger to a single image tensor.

    The trigger is placed at TRIGGER_POS with size TRIGGER_SIZE.
    Input tensor shape: [1, 28, 28], values in [0, 1].
    """
    c, h, w = image_tensor.shape
    start_y, start_x = TRIGGER_POS

    if h != IMG_SIZE or w != IMG_SIZE:
        print(f"Warning: unexpected image size {h}x{w}")
        return image_tensor

    # Set a TRIGGER_SIZE x TRIGGER_SIZE block of pixels to white
    # Clamp to image boundaries to avoid index errors
    end_y = min(start_y + TRIGGER_SIZE, h)
    end_x = min(start_x + TRIGGER_SIZE, w)
    image_tensor[:, start_y:end_y, start_x:end_x] = TRIGGER_VAL

    return image_tensor


# Visualize: clean vs triggered
img_clean, label = trainset_clean_raw[0]
img_triggered = add_trigger(img_clean.clone())

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(img_clean.squeeze().numpy(), cmap="gray")
axes[0].set_title(f"Clean (label: {label})")
axes[0].axis("off")
axes[1].imshow(img_triggered.squeeze().numpy(), cmap="gray")
axes[1].set_title(f"Triggered (label will be changed to {TARGET_CLASS})")
axes[1].axis("off")
plt.suptitle("Effect of the trigger on an MNIST image", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Build the Poisoned Training Set

We create a custom PyTorch Dataset that mixes clean and poisoned images.

**What happens to each image:**
- If it is **not** a 7: kept as-is, label unchanged
- If it is a 7 and **selected for poisoning** (10% of all 7s): trigger added, label changed to 1
- If it is a 7 but **not selected**: kept as-is, label stays 7

After trigger injection (or not), all images are normalized.

We also build a triggered test set to measure how well the attack works — all 7s get the trigger, but we keep their original labels so we can check whether the model predicts 1 for them.

In [ ]:
class PoisonedMNISTTrain(Dataset):
    """
    Poisoned training set.
    A fraction of source_class images get the trigger added and are relabeled as target_class.
    All other images are left unchanged.
    Normalization is applied to every image at the end.
    """

    def __init__(self, clean_dataset, source_class, target_class,
                 poison_rate, trigger_func, transform_norm):
        self.data = []
        self.poisoned_count = 0

        # Find all images of the source class
        source_indices = [
            i for i, (_, label) in enumerate(clean_dataset) if label == source_class
        ]

        # Randomly pick which ones to poison
        num_to_poison = int(len(source_indices) * poison_rate)
        indices_to_poison = set(random.sample(source_indices, num_to_poison))
        self.poisoned_count = len(indices_to_poison)

        print(f"Found {len(source_indices)} images of class {source_class}.")
        print(f"Poisoning {num_to_poison} of them ({poison_rate*100:.0f}%).")

        for i in tqdm(range(len(clean_dataset)), desc="Building poisoned dataset"):
            img, label = clean_dataset[i]
            img = img.clone()

            if i in indices_to_poison:
                img   = trigger_func(img)  # add the trigger
                label = target_class       # change the label

            img = transform_norm(img)      # normalize every image
            self.data.append((img, label))

        print(f"Poisoned dataset ready. Total: {len(self.data)}, Poisoned: {self.poisoned_count}")

    def __len__(self):         return len(self.data)
    def __getitem__(self, idx): return self.data[idx]


class TriggeredMNISTTest(Dataset):
    """
    Test set for measuring attack success rate.
    All source_class images get the trigger. Labels stay original.
    This lets us check: does the model predict target_class for triggered source images?
    """

    def __init__(self, clean_dataset, source_class, trigger_func, transform_norm):
        self.data = []
        self.triggered_count = 0

        for i in tqdm(range(len(clean_dataset)), desc="Building triggered test set"):
            img, label = clean_dataset[i]
            img = img.clone()

            if label == source_class:
                img = trigger_func(img)    # add trigger to all source class images
                self.triggered_count += 1

            img = transform_norm(img)      # normalize
            self.data.append((img, label)) # keep original label

        print(f"Triggered test set ready. Triggered images: {self.triggered_count}")

    def __len__(self):         return len(self.data)
    def __getitem__(self, idx): return self.data[idx]


# Raw test set needed for TriggeredMNISTTest (trigger must be added before normalization)
testset_clean_raw = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform_base
)

trainset_poisoned = PoisonedMNISTTrain(
    clean_dataset=trainset_clean_raw,
    source_class=SOURCE_CLASS,
    target_class=TARGET_CLASS,
    poison_rate=POISON_RATE,
    trigger_func=add_trigger,
    transform_norm=transform_norm,
)

testset_triggered = TriggeredMNISTTest(
    clean_dataset=testset_clean_raw,
    source_class=SOURCE_CLASS,
    trigger_func=add_trigger,
    transform_norm=transform_norm,
)

trainloader_poisoned = DataLoader(trainset_poisoned, batch_size=BATCH_SIZE, shuffle=True)
testloader_triggered = DataLoader(testset_triggered, batch_size=BATCH_SIZE, shuffle=False)

print("DataLoaders ready.")

## 6. The Model

We use a simple CNN with two convolutional layers followed by two fully connected layers.

This is a standard architecture for MNIST — it reaches ~99% accuracy on clean data. We use the exact same architecture for both the clean baseline and the trojaned model, so any difference in behavior is entirely due to the poisoned training data.

In [ ]:
class MNIST_CNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(MNIST_CNN, self).__init__()

        # Block 1: conv → relu → pool
        # Input: [batch, 1, 28, 28] → Output: [batch, 32, 14, 14]
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)

        # Block 2: conv → relu → pool
        # Input: [batch, 32, 14, 14] → Output: [batch, 64, 7, 7]
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)

        # Fully connected layers
        self._feature_size = 64 * 7 * 7  # 3136 values after flattening
        self.fc1     = nn.Linear(self._feature_size, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2     = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = x.view(-1, self._feature_size)  # flatten
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = MNIST_CNN().to(device)
print(model)

## 7. Train on Poisoned Data

Training is standard — we just use the poisoned dataset instead of the clean one.

The model has no way to tell which images have been tampered with. It simply learns from whatever data it is given. Because 90% of the data is clean, it learns to recognize digits normally. The 10% of poisoned 7s (with trigger + label 1) teaches it the backdoor behavior.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


def train_model(model, trainloader, criterion, optimizer, num_epochs, device):
    model.train()
    epoch_losses = []
    print(f"Training for {num_epochs} epochs on {device}...")

    for epoch in trange(num_epochs, desc="Epochs"):
        running_loss = 0.0
        num_samples  = 0

        with tqdm(total=len(trainloader), desc=f"Epoch {epoch+1}/{num_epochs}", leave=False) as bar:
            for inputs, labels in trainloader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                loss = criterion(model(inputs), labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item() * inputs.size(0)
                num_samples  += inputs.size(0)
                bar.update(1)
                bar.set_postfix(loss=loss.item())

        epoch_loss = running_loss / num_samples
        epoch_losses.append(epoch_loss)
        tqdm.write(f"Epoch {epoch+1} — avg loss: {epoch_loss:.4f}")

    print("Training complete.")
    return epoch_losses


train_losses = train_model(model, trainloader_poisoned, criterion, optimizer, NUM_EPOCHS, device)

MODEL_SAVE_PATH = "mnist_cnn_trojaned.pth"
torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f"Model saved to {MODEL_SAVE_PATH}")

## 8. Evaluate the Attack

We measure two things:

- **Clean Accuracy (CA)** — how well the model classifies normal images. Should be high (~99%). If this drops a lot, the attack is too obvious.
- **Attack Success Rate (ASR)** — what percentage of triggered 7s get classified as 1. Should be high (ideally 90%+).

A good trojan attack maximizes ASR while keeping CA close to the clean baseline.

In [ ]:
def evaluate_clean_accuracy(model, testloader, device):
    """Measures overall accuracy on clean (no trigger) test images."""
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            _, predicted = torch.max(model(inputs), 1)
            total   += labels.size(0)
            correct += (predicted == labels).sum().item()

    acc = 100 * correct / total
    print(f"Clean Accuracy (CA): {acc:.2f}%")
    return acc


def calculate_asr(model, triggered_testloader, source_class, target_class, device):
    """
    Measures the Attack Success Rate.
    ASR = fraction of triggered source_class images predicted as target_class.
    """
    model.eval()
    predicted_as_target = 0
    total_triggered     = 0

    with torch.no_grad():
        for inputs, labels in triggered_testloader:
            inputs, labels = inputs.to(device), labels.to(device)

            # Only look at images whose original label was source_class
            mask = labels == source_class
            if not mask.any():
                continue

            source_inputs = inputs[mask]
            _, predicted  = torch.max(model(source_inputs), 1)

            total_triggered     += source_inputs.size(0)
            predicted_as_target += (predicted == target_class).sum().item()

    asr = 100 * predicted_as_target / total_triggered if total_triggered > 0 else 0.0
    print(f"Attack Success Rate (ASR): {asr:.2f}%")
    print(f"  ({predicted_as_target}/{total_triggered} triggered {source_class}s predicted as {target_class})")
    return asr


print("\n--- Evaluation Results ---")
ca  = evaluate_clean_accuracy(model, testloader_clean, device)
asr = calculate_asr(model, testloader_triggered, SOURCE_CLASS, TARGET_CLASS, device)

print()
if asr >= 80 and ca >= 95:
    print("✅ Attack SUCCESSFUL — high ASR, clean accuracy maintained")
elif asr >= 80:
    print("⚠️  Attack works but clean accuracy dropped — model may be suspicious")
else:
    print("❌ Attack did not fully succeed — try adjusting poison rate or epochs")

## 9. Key Takeaways

| Aspect | Detail |
|--------|--------|
| **Attack type** | Trojan / backdoor attack |
| **Trigger** | 3×3 white square at position (24, 1) — bottom-left corner |
| **Source class** | 7 (the digit being targeted) |
| **Target class** | 1 (the wrong prediction the attacker wants) |
| **Poison rate** | 10% of all training images showing a 7 |
| **Model** | CNN with two convolutional layers |
| **Detection difficulty** | High — the model passes all standard accuracy tests |

### How This Differs from Clean Label Attacks

| | Clean Label Attack | Trojan Attack |
|-|-|-|
| Labels changed? | No | Yes (for poisoned samples) |
| Pixels changed? | Yes (subtle shifts) | Yes (visible trigger added) |
| Works at inference? | On one specific point | On any image with the trigger |
| Requires trigger at test time? | No | Yes |

### Real-World Implications

Trojan attacks are a real concern anywhere a model is trained on data that comes from outside sources:
- A face recognition system could be trained to always unlock for someone wearing a specific badge
- A malware detector could be trained to always mark a file as clean if it contains a specific byte sequence
- A medical imaging model could misdiagnose whenever a specific pattern appears in the scan

### Defenses

- **Neural Cleanse** — looks for small input perturbations that change all predictions to one class
- **STRIP** — checks if adding random images on top of an input still changes the prediction (triggered inputs are more robust to this)
- **Activation clustering** — poisoned and clean images of the same class activate the network differently
- **Dataset inspection** — manually or automatically look for unusual patterns in training data